In [1]:
from threading import Thread

import rclpy
from rclpy.callback_groups import ReentrantCallbackGroup
from rclpy.node import Node
from commander_py import commander
from commander_py import commander_utils
from commander_py import ros_tf_utils
from commander_py import transform_utils
from geometry_msgs.msg import Pose, Point, Quaternion
from ur_commander.srv import VisualizePoses

import tf2_ros
from rclpy.time import Time


import numpy as np

# Initialize rclpy, handling already initialized case
try:
    rclpy.init()
except RuntimeError:
    pass

node = Node("ex_pose_goal")
callback_group = ReentrantCallbackGroup()
commander = commander.Commander(
    node=node,
    callback_group=callback_group,
    move_group="ur_manipulator",
    end_effector_frame=["camera_color_optical_frame", "tcp_ee"],
)
executor = rclpy.executors.MultiThreadedExecutor(2)
executor.add_node(node)
executor_thread = Thread(target=executor.spin, daemon=True)
executor_thread.start()
node.create_rate(1.0).sleep()

[INFO] [1753689839.842458405] [ex_pose_goal]: UR Commander Node Initialized


In [2]:
def display_poses(poses: list[Pose], frmae_id: str = "base_link") -> None:

    client = node.create_client(VisualizePoses, "/commander_viz/pose_visualization")
    while not client.wait_for_service(timeout_sec=3.0):
        node.get_logger().info("Waiting for /commander_viz/pose_visualization service...")
    client.call_async(VisualizePoses.Request(poses=poses, frame_id=frmae_id))

In [3]:
target = Pose(
    position=Point(x=0.5, y=0.0, z=0.5), orientation=Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)
)

success = display_poses([target], "base_link")
if success:
    node.get_logger().info("Pose visualization request sent successfully.")

In [4]:
target = commander_utils.rotate_pose_euler(target, euler_deg=(0, 0, 270))
target = commander_utils.rotate_pose_euler(target, euler_deg=(180, 0, 0))
target_1 = commander_utils.translate_pose(target, translation=(0.0, 0.4, 0.0))
success = display_poses([target, target_1], "base_link")
# success = display_poses([target], "base_link")

In [5]:
joint_values = [0.0, -1.57, 1.57, -1.57, -1.57, 0.0]
goal = commander.set_joint_goal(joint_values=joint_values,ee_link="tcp_ee")

In [6]:
goal = commander.set_pose_goal(
    pose=target, frame_id="base_link", ee_link="tcp_ee"
)
print(goal)

moveit_msgs.msg.MotionPlanRequest(workspace_parameters=moveit_msgs.msg.WorkspaceParameters(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=0, nanosec=0), frame_id='base_link'), min_corner=geometry_msgs.msg.Vector3(x=-1.0, y=-1.0, z=-1.0), max_corner=geometry_msgs.msg.Vector3(x=1.0, y=1.0, z=1.0)), start_state=moveit_msgs.msg.RobotState(joint_state=sensor_msgs.msg.JointState(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1753689852, nanosec=243598071), frame_id='base_link'), name=['shoulder_lift_joint', 'elbow_joint', 'wrist_1_joint', 'wrist_2_joint', 'wrist_3_joint', 'shoulder_pan_joint'], position=[-1.57, 0.0, -1.57, 0.0, 0.0, 0.0], velocity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], effort=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]), multi_dof_joint_state=sensor_msgs.msg.MultiDOFJointState(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=0, nanosec=0), frame_id=''), joint_names=[], transforms=[], twist=[], wrench=[]), attached_collision_objects=[], is_diff

In [7]:
traj = commander.plan(
    pose_goal=goal,
    # joint_goal=goal,
    planner_id="PTP",
    pipeline_id="pilz_industrial_motion_planner",
    ee_frame="tcp_ee",
    acc_scale=0.2,
    vel_scale=0.2,
)

[INFO] [1753689854.755726212] [ex_pose_goal]: Planning successful


In [8]:
success = commander.execute_trajectory(traj, wait_until_executed=True)
if success:
    node.get_logger().info("Trajectory executed successfully.")

[INFO] [1753689857.033426742] [ex_pose_goal]: Sending goal to action server /execute_trajectory with trajectory
[INFO] [1753689857.034438400] [ex_pose_goal]: Goal accepted by action server /execute_trajectory
[INFO] [1753689862.140735837] [ex_pose_goal]: Execution completed
[INFO] [1753689862.145421929] [ex_pose_goal]: Trajectory executed successfully.


In [11]:
sequence_goals = [
    commander.set_pose_goal(pose=target, frame_id="base_link", ee_link="tcp_ee"),
    commander.set_pose_goal(pose=target_1, frame_id="base_link", ee_link="tcp_ee"),
    commander.set_joint_goal(joint_values=joint_values),
]
traj_sequence = commander.plan_sequence(
    pose_goals=sequence_goals,
    pipeline_id="pilz_industrial_motion_planner",
    planner_ids=["PTP", "LIN", "PTP"],
    ee_frame="tcp_ee",
    acc_scale=0.2,
    vel_scale=0.2,
)

[INFO] [1753689876.245243278] [ex_pose_goal]: Planning sequence successful


In [12]:
success = commander.execute_trajectory(traj_sequence, wait_until_executed=True)

[INFO] [1753689878.840099488] [ex_pose_goal]: Sending goal to action server /execute_trajectory with trajectory
[INFO] [1753689878.841232203] [ex_pose_goal]: Goal accepted by action server /execute_trajectory
[INFO] [1753689882.697314773] [ex_pose_goal]: Execution completed


In [13]:
tf_buffer, tf_listener = ros_tf_utils.create_tf_buffer_and_listener(node)

In [14]:
# Give time to receive at least one TF message
for _ in range(20):
    rclpy.spin_once(node, timeout_sec=0.1)

tr = ros_tf_utils.get_transform(tf_buffer, "base_link", "tcp_ee", logger=node.get_logger())
print(tr)

if tr:
    T = ros_tf_utils.get_transform_matrix(tf_buffer, "base_link", "tcp_ee", logger=node.get_logger())
    print(T)


geometry_msgs.msg.TransformStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1753689889, nanosec=405616048), frame_id='base_link'), child_frame_id='tcp_ee', transform=geometry_msgs.msg.Transform(translation=geometry_msgs.msg.Vector3(x=0.6891375393334018, y=0.17285033243621184, z=0.5417353616142425), rotation=geometry_msgs.msg.Quaternion(x=-0.7071065569129826, y=0.7071067812590627, z=0.0005630880171417206, w=-1.4502998800949098e-10)))
[[-6.34341334e-07 -9.99999683e-01 -7.96326663e-04  6.89137539e-01]
 [-9.99999683e-01  2.05103760e-10  7.96326506e-04  1.72850332e-01]
 [-7.96326253e-04  7.96326916e-04 -9.99999366e-01  5.41735362e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
